# 變異prompt
設定初始的prompt 判斷漲跌並計算其年化  
再拿最高的年化報酬率的prompt變異

In [1]:
# %pip install numpy_financial
# %pip install pandas
# %pip install numpy
# %pip install langchain_google_genai

In [2]:
import os
import textwrap
from IPython.display import display
from IPython.display import Markdown
from datetime import datetime
import numpy_financial as npf
import math
import pandas as pd
import numpy as np

from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory
from langchain_google_genai import ChatGoogleGenerativeAI

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = os.getenv('GEMINI_API_KEY')

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))


c:\Users\user\Desktop\LLM-Predict-Stock\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# p_mutate = '創新分析及其對預期市場影響的評估，表明股價持續上漲，反映了投資者對公司令人振奮的長期增長前景的樂觀情緒。'

# prompt = '請對於台積電' + p_mutate + '回答 #非常好 #好 #無關 #不好 #非常不好'
# news = '''台積電執行副總經理暨共同營運長米玉傑在聖克拉拉（Santa Clara）的會議上表示，該技術將得以從晶圓背面向運算晶片輸送電力，有助於加快人工智慧（AI）晶片的速度。'''

# llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=1)
# response = llm.invoke(prompt  + news)
# response.content

In [4]:
def gemini_response(model, instruction, count):
    text = ''
    for i in range(count):
        try:
            result = model.invoke(instruction)
            text = result.content
        except:
            pass
    # print(text)

    if text.find('非常不好') > 0:
        score = -2
    elif text.find('非常好') > 0:
        score = 2
    elif text.find('不好') > 0:
        score = -1
    elif text.find('好') > 0:
        score = 1
    else:
        score = 0

    return score  # 分數

In [5]:
def calculate_return(signal, threshold):
    # signal = [日期, 評分, 價格]
    hold = 0  # 持有的股數
    balance = 10000000
    init_balance = balance

    for item in signal:
        score = float(item[1])
        price = float(item[2])

        if hold == 0:
            if score == 2:
                hold += int(balance/price)
                balance = 0
            elif score == 1:
                hold += int((balance/2)/price)
                balance = int(balance/2)
        elif hold > 0:
            if score < 0:
                balance += price*hold
                hold = 0
            elif score == 2:
                hold += int(balance/price)
                balance = 0
    
    balance += hold*price
    last_price = float(signal[len(signal) - 1][2])
    bnh = last_price / float(signal[0][2])
    r = balance/init_balance

    return round(r, 5), round(bnh, 5)

In [6]:
# def calculate_return(signal, threshold):
#     # signal = [日期, 評分, 價格]
#     hold = 0  # 持有的股數
#     balance = 10000000
#     init_balance = balance

#     for item in signal:
#         score = float(item[1])
#         price = float(item[2])

#         if hold == 0:
#             if score>0:
#                 hold = int(balance/price)
#         elif hold > 0:
#             if score < 0:
#                 balance += price*hold
#                 hold = 0
    
#     balance += hold*price
#     last_price = float(signal[len(signal) - 1][2])
#     bnh = last_price / float(signal[0][2])
#     r = balance/init_balance

#     return round(r, 5), round(bnh, 5)

In [7]:
def mutate_prompt(llm, p_mutate):
    prompt = '產生以下指令的1個變體，同時保留語意：'
    response = llm.invoke(prompt + p_mutate)
    return response.content

In [8]:
# main cell

regenerate_count = 1
prompt_mutate_count = 100

start_time = datetime(2022, 1, 1)
end_time = datetime(2023, 12, 31)

stock_ids = ["1101", "2211", "2385", "2542", "2880", "2912", "3023", "3264", "5269", "8027"]
stock_names = ["台泥", "長榮鋼", "群光", "興富發", "華南金", "統一超", "信邦", "欣銓", "祥碩", "鈦昇"]
# stock_ids = ["2912", "3023", "3264", "5269", "8027"]
# stock_names = ["統一超", "信邦", "欣銓", "祥碩", "鈦昇"]
# stock_ids = ["8027"]
# stock_names = ["鈦昇"]

llm = ChatGoogleGenerativeAI(
    model="gemini-pro",
    temperature=1,
    safety_settings={
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
    })

p_mutate = '深入探討業務創新策略的潛在影響，仔細評估預期對目標受眾的影響。仔細考察投資者對增長的預測，探討這些預測如何影響股票的持續增長。此外，徹底了解支持這些假設的基礎驅動因素。'
rtn_ = 1.190583

for i in range(79,prompt_mutate_count):
    # 變異
    p_mutate_2 = mutate_prompt(llm, p_mutate)

    rtns = []
    bnhs = []
    raw_data = []
    for idx, stock_id in enumerate(stock_ids):
        print(f'變異{i + 1}次\t執行{stock_id} {stock_names[idx]}')
        general_prompt = '請對於'+ stock_names[idx] +'，' + p_mutate_2 + '回答 #非常好 #好 #無關 #不好 #非常不好'
        folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
        files = os.listdir(folder_path)
        sorted_files = sorted(files)

        # 載入買進價格
        df = pd.read_csv(os.path.dirname(os.path.abspath(os.getcwd())) + "/price_history/" + stock_id + ".csv", dtype=str)
        price_np = df.to_numpy()
        prices = price_np.tolist()  # [時間,價格]

        signal_2 = []

        for price in prices:
            # 檢查日期是否在參考日期之後
            try:
                date_obj = datetime.strptime(price[0], '%Y%m%d')
            except:
                continue

            if not (start_time <= date_obj <= end_time):
                continue
            
            sig = 0
            for index, file_name in enumerate(sorted_files):
                if file_name == price[0]:
                    with open(folder_path + file_name, encoding='utf-8') as f:
                        news = f.read()
                        # print(news)
                        prompt = general_prompt + "\n" + news
                        # gemini的回應，回傳帶有買入訊號的list
                        sig = gemini_response(llm, prompt, regenerate_count)
                        # print(sig, prompt)
                    break
            
            if sig != 0 :
                signal_2.append([price[0], sig, price[1]])

        signal_2_np = np.array(signal_2)
        # 回傳 IRR 以及總額投入
        rtn, bnh = calculate_return(signal_2_np, 0)
        rtns.append(rtn)
        bnhs.append(bnh)
        raw_data.append(signal_2_np.tolist())
        # print("rtn", irr)
        # print("總額投入報酬率", bnh)

    # 輸出結果
    write_folder = "eval_gemini/output"
    count = 1
    while True:
        if os.path.exists(write_folder + str(count)):
            count += 1
            continue
        else:
            with open(write_folder + str(count), "a", encoding="UTF-8") as f:
                content = (f'提示：{p_mutate_2}\n mutate from {p_mutate}\n'
                           f'開始日期 {start_time.strftime("%Y%m%d")},'
                           f'結束日期 {end_time.strftime("%Y%m%d")}\n')
                for j in range(len(stock_ids)):
                    content += (f'股票代號：{stock_ids[j]}\t'
                                f'依照分析報酬率:{rtns[j]}\t'
                                f'總額投入報酬率:{bnhs[j]}\n')
                content += f'變異{i + 1}次，平均報酬率:{sum(rtns) / len(rtns)}，平均總額投入報酬率:{1.186132}\n)'
                content += f'原始數據{raw_data}'
                f.write(content)
            print("寫入" + write_folder + str(count))
            break

    # 判斷是否更改變異
    if sum(rtns) / len(rtns) > rtn_:
        print("更改變異提示為:", p_mutate_2)
        p_mutate = p_mutate_2
        rtn_ = sum(rtns) / len(rtns)

# todo testing
